# Cross-Validation for Binary Classification

This notebook focuses on **cross-validation** using the dataset:

`binary_classification_3_features.csv`

The dataset contains:

- `feature_1`
- `feature_2`
- `feature_3`
- `target` — binary class: `0` or `1`

We will cover:

1. Loading and inspecting the dataset
2. Why cross-validation is useful
3. Stratified K-Fold cross-validation
4. `cross_val_score`
5. Evaluating multiple metrics with `cross_validate`
6. Inspecting scores fold by fold
7. Comparing models with cross-validation
8. Hyperparameter tuning with `GridSearchCV`


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score,
    cross_validate,
    GridSearchCV
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score

RANDOM_STATE = 42


## 1. Load the dataset

Place the CSV file in the **same directory as this notebook**.

If you are running this notebook in the environment where it was generated,
the fallback path `/mnt/data/binary_classification_3_features.csv` is also checked.


In [ ]:


df = pd.read_csv("./binary_classification_3_features.csv")

print("Shape:", df.shape)

df.head()


In [ ]:
df.info()

print("\nClass counts:")
print(df["target"].value_counts().sort_index())

print("\nClass proportions:")
print(df["target"].value_counts(normalize=True).sort_index())


## 2. Separate features and target

`X` contains the three input features.

`y` contains the binary label.


In [ ]:
X = df[["feature_1", "feature_2", "feature_3"]]
y = df["target"]

print("X shape:", X.shape)
print("y shape:", y.shape)


## 3. Why use cross-validation?

A single train/test split can give a result that depends strongly on which samples
happen to enter the training set and which samples enter the test set.

With **K-Fold Cross-Validation**:

1. Split the dataset into `K` folds.
2. Train on `K-1` folds.
3. Validate on the remaining fold.
4. Repeat until every fold has been used once as validation data.
5. Average the validation scores.

For classification, **Stratified K-Fold** is usually preferred because it preserves
approximately the same class proportions in every fold.


In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

for fold, (train_idx, val_idx) in enumerate(cv.split(X, y), start=1):
    y_train_fold = y.iloc[train_idx]
    y_val_fold = y.iloc[val_idx]

    print(f"Fold {fold}")
    print(f"  Training samples:   {len(train_idx)}")
    print(f"  Validation samples: {len(val_idx)}")
    print(
        "  Train class distribution:",
        y_train_fold.value_counts(normalize=True).sort_index().round(3).to_dict()
    )
    print(
        "  Valid class distribution:",
        y_val_fold.value_counts(normalize=True).sort_index().round(3).to_dict()
    )
    print()


## 4. Cross-validation with Logistic Regression

We use a `Pipeline` so that scaling is fitted **inside each training fold**.

This is important: fitting the scaler on the full dataset before cross-validation
would leak information from the validation folds into the training process.


In [ ]:
logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000))
])

accuracy_scores = cross_val_score(
    logistic_model,
    X,
    y,
    cv=cv,
    scoring="accuracy"
)

print("Accuracy for each fold:")
for i, score in enumerate(accuracy_scores, start=1):
    print(f"Fold {i}: {score:.4f}")

print(f"\nMean accuracy: {accuracy_scores.mean():.4f}")
print(f"Standard deviation: {accuracy_scores.std():.4f}")


In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(range(1, len(accuracy_scores) + 1), accuracy_scores)
plt.axhline(
    accuracy_scores.mean(),
    linestyle="--",
    label=f"Mean = {accuracy_scores.mean():.3f}"
)
plt.xlabel("Fold")
plt.ylabel("Accuracy")
plt.title("5-Fold Cross-Validation Accuracy")
plt.ylim(0, 1)
plt.legend()
plt.show()


## 5. Evaluate several metrics

Accuracy alone may not always tell the full story.

For binary classification, useful metrics include:

- **accuracy**
- **precision**
- **recall**
- **F1 score**

`cross_validate` can compute several metrics during the same cross-validation run.


In [ ]:
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
}

results = cross_validate(
    logistic_model,
    X,
    y,
    cv=cv,
    scoring=scoring,
    return_train_score=True
)

results_df = pd.DataFrame({
    "fold": np.arange(1, 6),
    "train_accuracy": results["train_accuracy"],
    "valid_accuracy": results["test_accuracy"],
    "precision": results["test_precision"],
    "recall": results["test_recall"],
    "f1": results["test_f1"],
})

results_df


In [ ]:
print("Average validation metrics:")

for metric in ["accuracy", "precision", "recall", "f1"]:
    values = results[f"test_{metric}"]
    print(
        f"{metric:10s}: "
        f"{values.mean():.4f} ± {values.std():.4f}"
    )


## 6. Manual cross-validation loop

Scikit-learn functions such as `cross_val_score` are convenient, but it is useful
to see what is happening internally.

The following loop explicitly:

1. Creates each train/validation split
2. Fits the model
3. Predicts validation labels
4. Computes accuracy


In [ ]:
from sklearn.metrics import accuracy_score

manual_scores = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X, y), start=1):
    X_train = X.iloc[train_idx]
    X_val = X.iloc[val_idx]

    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=1000))
    ])

    model.fit(X_train, y_train)

    predictions = model.predict(X_val)

    score = accuracy_score(y_val, predictions)
    manual_scores.append(score)

    print(f"Fold {fold}: accuracy = {score:.4f}")

print(f"\nMean accuracy = {np.mean(manual_scores):.4f}")


## 7. Compare multiple models

Cross-validation is especially useful when comparing different algorithms.

We will compare:

- Logistic Regression
- Decision Tree
- K-Nearest Neighbors

Each model is evaluated using the **same cross-validation strategy**.


In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=1000))
    ]),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=4,
        random_state=RANDOM_STATE
    ),

    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", KNeighborsClassifier(n_neighbors=7))
    ])
}

model_results = []

for name, model in models.items():
    scores = cross_val_score(
        model,
        X,
        y,
        cv=cv,
        scoring="accuracy"
    )

    model_results.append({
        "model": name,
        "mean_accuracy": scores.mean(),
        "std_accuracy": scores.std()
    })

comparison_df = pd.DataFrame(model_results)
comparison_df.sort_values("mean_accuracy", ascending=False)


In [ ]:
plt.figure(figsize=(8, 4))

plt.bar(
    comparison_df["model"],
    comparison_df["mean_accuracy"],
    yerr=comparison_df["std_accuracy"],
    capsize=5
)

plt.ylabel("Mean Cross-Validation Accuracy")
plt.title("Model Comparison with 5-Fold Cross-Validation")
plt.ylim(0, 1)
plt.xticks(rotation=15)
plt.show()


## 8. Hyperparameter tuning with GridSearchCV

Cross-validation can also be used to choose model hyperparameters.

Here we tune the regularization parameter `C` of Logistic Regression.

Smaller `C` means stronger regularization.

Larger `C` means weaker regularization.


In [ ]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000))
])

param_grid = {
    "classifier__C": [0.01, 0.1, 1, 10, 100]
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="accuracy",
    return_train_score=True
)

grid_search.fit(X, y)

print("Best C:", grid_search.best_params_["classifier__C"])
print(f"Best mean CV accuracy: {grid_search.best_score_:.4f}")


In [ ]:
grid_results = pd.DataFrame(grid_search.cv_results_)

grid_results[
    [
        "param_classifier__C",
        "mean_train_score",
        "mean_test_score",
        "std_test_score"
    ]
].sort_values("mean_test_score", ascending=False)


In [ ]:
plt.figure(figsize=(7, 4))

plt.semilogx(
    grid_results["param_classifier__C"].astype(float),
    grid_results["mean_test_score"],
    marker="o"
)

plt.xlabel("C")
plt.ylabel("Mean Cross-Validation Accuracy")
plt.title("Logistic Regression Hyperparameter Search")
plt.ylim(0, 1)
plt.grid(True)
plt.show()


## 9. Important cross-validation lessons

### Use stratification for classification

`StratifiedKFold` keeps the class proportions approximately equal across folds.

### Preprocessing must happen inside the fold

Use a `Pipeline` for operations such as:

- scaling
- normalization
- PCA
- feature selection

This prevents **data leakage**.

### Look at both mean and standard deviation

For example:

```text
accuracy = 0.91 ± 0.03
```

The mean tells us the typical performance.

The standard deviation tells us how much performance changes across folds.

### Cross-validation does not replace a final test set

In a real machine-learning project, a common workflow is:

1. Keep a final test set untouched.
2. Use cross-validation on the training data.
3. Select the model and hyperparameters.
4. Retrain using all training data.
5. Evaluate exactly once on the final test set.


## Exercises

Try changing:

```python
n_splits=5
```

to:

```python
n_splits=10
```

and compare the mean and standard deviation.

You can also try:

- different Decision Tree depths
- different KNN values
- `scoring="f1"` instead of accuracy
- `RepeatedStratifiedKFold`
- adding a final held-out test set
